In [1]:
%load_ext autoreload
%autoreload 2
import os
home_dir = os.path.expanduser('~')
os.chdir(os.path.join(home_dir, "DeepFact"))

## Up to date save func

In [3]:
import os
import json
import math
import pandas as pd
import numpy as np

# -------------------- Config --------------------
model_names = [
    "gemini-2.5-pro-deepresearch",
    "tavily-research",
    "openai-deepresearch",
    # "kimi-researcher",
    # "claude-research",
    # "langchain-open-deep-research",
]

# Numeric score columns we’ll try to extract for the leaderboard
SCORE_COLUMNS = [
    "overall_score",
    "comprehensiveness_score",
    "insight_score",
    "instruction_following_score",
    "readability_score",
]

DATA_URL = "https://huggingface.co/spaces/Ayanami0730/DeepResearch-Leaderboard/resolve/main/data/data_viewer.jsonl"

# -------------------- Load once --------------------
# Load the large combined file (has `article`) and rename columns:
#   article -> response, prompt -> question
df_all = pd.read_json(DATA_URL, lines=True)
df_all = df_all.rename(columns={"article": "response", "prompt": "question"})

# Figure out the right model-name column
_MODEL_COL = None
for cand in ["model_name", "model", "agent", "agent_name", "run_model"]:
    if cand in df_all.columns:
        _MODEL_COL = cand
        break

if _MODEL_COL is None:
    raise RuntimeError(
        "Could not locate a model name column in data_viewer.jsonl. "
        "Checked: model_name, model, agent, agent_name, run_model."
    )

# -------------------- Helpers --------------------
def _subset_last50_for_model(model: str) -> pd.DataFrame:
    """Get English-filtered last 50 rows for a model from the global DF."""
    dfm = df_all[df_all[_MODEL_COL].astype(str).str.strip().eq(model)].copy()

    # Prefer English if column exists
    if "language" in dfm.columns:
        df_en = dfm[dfm["language"].astype(str).str.lower().eq("en")]
        if not df_en.empty:
            dfm = df_en

    # Keep last 50 (or fewer)
    # return dfm
    dfm = dfm.sort_values('id')

    return dfm.tail(min(50, len(dfm))).copy()

def _expand_score_dicts(df: pd.DataFrame) -> pd.DataFrame:
    """If score columns are nested in a dict col, expand them into flat columns."""
    if set(SCORE_COLUMNS).issubset(df.columns):
        return df  # Already flat

    dict_col = None
    for cand in ["scores", "rubric_scores", "metrics", "evaluation", "score_breakdown"]:
        if cand in df.columns and df[cand].apply(lambda x: isinstance(x, dict)).any():
            dict_col = cand
            break
    if dict_col:
        expanded = pd.json_normalize(df[dict_col])
        for c in expanded.columns:
            if c not in df.columns:
                df[c] = expanded[c]
    return df

def load_last50_scores(model: str) -> pd.DataFrame:
    """Return numeric-only score frame for averaging/leaderboard."""
    df = _subset_last50_for_model(model)
    df = _expand_score_dicts(df)

    present_cols = [c for c in SCORE_COLUMNS if c in df.columns]
    for c in present_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df[present_cols].copy()

def _to_builtin(obj):
    """Recursively convert pandas/numpy types to JSON-safe Python types; NaN/Inf -> None."""
    if isinstance(obj, dict):
        return {k: _to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_builtin(v) for v in obj]
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    return obj

def save_last50_reports(model: str, out_root: str = "data/deep_research_bench") -> None:
    """
    Save each of the (English-filtered) last 50 rows for `model` as:
      data/deep_research_bench/{model}/drb_{report_idx}.json

    The loaded DF already uses:
      - 'response' instead of 'article'
      - 'question' instead of 'prompt'
    """
    dfm = _subset_last50_for_model(model)
    out_dir = os.path.join(out_root, model)
    os.makedirs(out_dir, exist_ok=True)

    # Reorder by original order; enumerate deterministically
    dfm = dfm.reset_index(drop=True)

    for idx, row in dfm.iterrows():
        base = f"drb_{idx}"  # change to f"drb_{idx + 1:03d}" if you want zero padding
        out_path = os.path.join(out_dir, f"{base}.json")

        # Create a JSON-safe dict
        row_dict = _to_builtin(row.where(pd.notnull(row), None).to_dict())

        # Add convenience fields
        row_dict["_model"] = model
        row_dict["_report_name"] = base
        row_dict["_report_idx"] = idx

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(row_dict, f, ensure_ascii=False, indent=2)

    print(f"[{model}] Saved {len(dfm)} reports to {out_dir}")
    return dfm

# -------------------- Leaderboard --------------------
rows = {}
for m in model_names:
    df_scores = load_last50_scores(m)
    if df_scores.empty:
        rows[m] = pd.Series(index=SCORE_COLUMNS, dtype="float64")
    else:
        rows[m] = df_scores.mean(numeric_only=True)

final_df = pd.DataFrame.from_dict(rows, orient="index")
final_df = final_df.reindex(columns=SCORE_COLUMNS)

if "overall_score" in final_df.columns:
    final_df = final_df.sort_values("overall_score", ascending=False)

final_df = final_df.round(3)
print(final_df)

# -------------------- Save per-report JSONs --------------------
for m in model_names:
    dfm = save_last50_reports(m)


                             overall_score  comprehensiveness_score  \
tavily-research                     52.305                   52.780   
gemini-2.5-pro-deepresearch         49.859                   49.884   
openai-deepresearch                 47.005                   46.964   

                             insight_score  instruction_following_score  \
tavily-research                     54.069                       51.154   
gemini-2.5-pro-deepresearch         49.866                       49.770   
openai-deepresearch                 44.622                       49.463   

                             readability_score  
tavily-research                         48.348  
gemini-2.5-pro-deepresearch             50.041  
openai-deepresearch                     47.815  
[gemini-2.5-pro-deepresearch] Saved 50 reports to data/deep_research_bench/gemini-2.5-pro-deepresearch
[tavily-research] Saved 50 reports to data/deep_research_bench/tavily-research
[openai-deepresearch] Saved 50 repor

In [21]:
dfm

,model_name,id,question,response,overall_score,comprehensiveness_score,insight_score,instruction_following_score,readability_score
0,openai-deepresearch,51,"From 2020 to 2050, how many elderly people wil...",# Market Size Analysis of Japan’s Elderly Cons...,46.66,45.74,44.25,49.62,48.66
1,openai-deepresearch,52,What are the investment philosophies of Duan Y...,# Comparing the Investment Philosophies of Dua...,44.55,43.98,41.36,50.00,47.09
2,openai-deepresearch,53,Researching how the world's wealthiest governm...,# How the World’s Wealthiest Governments Inves...,48.19,45.68,48.99,49.49,49.51
3,openai-deepresearch,54,"In the field of FinTech, machine learning algo...",# Traditional vs. AI-Driven Models in Asset A...,48.26,47.66,48.24,49.30,47.55
4,openai-deepresearch,55,While the market features diverse quantitative...,# Evaluation Framework for Quantitative Tradi...,45.01,43.97,42.92,48.45,47.87
5,openai-deepresearch,56,Is there a general method for solving a first-...,# Solution Methods for Asymmetric First-Pr...,46.63,46.33,45.14,50.00,46.61
6,openai-deepresearch,57,"Summarize the global investments, key initiati...",# Global AI Initiatives by Leading Consult...,46.49,53.69,25.22,53.08,42.33
7,openai-deepresearch,58,Exploring Horizontal Gene Transfer (HGT) in Pl...,# Horizontal Gene Transfer in Plants and Anima...,48.68,49.39,46.41,50.00,52.28
8,openai-deepresearch,59,"In ecology, how do birds achieve precise locat...","# Bird Migration and Navigation: Mechanisms, C...",44.37,45.06,39.23,49.17,47.15
9,openai-deepresearch,60,How to conduct comprehensive and accurate situ...,# Situational Awareness of Cislunar Space Targ...,44.16,43.35,44.04,44.10,46.21
